# Week 11 exercises

### Ex 11.1

In [2]:
import numpy as np
import cv2

K = np.loadtxt('K.txt')
im0 = cv2.imread('sequence/000001.png')
im1 = cv2.imread('sequence/000002.png')
im2 = cv2.imread('sequence/000003.png')

In [3]:
sift = cv2.SIFT_create(nfeatures=2000)

kp0, des0 = sift.detectAndCompute(im0, None)
kp1, des1 = sift.detectAndCompute(im1, None)
kp2, des2 = sift.detectAndCompute(im2, None)

kp0 = np.array([k.pt for k in kp0])
kp1 = np.array([k.pt for k in kp1])
kp2 = np.array([k.pt for k in kp2])

bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)

matches01 = bf.match(des0, des1)
matches12 = bf.match(des1, des2)

matches01 = np.array([(m.queryIdx, m.trainIdx) for m in matches01])
matches12 = np.array([(m.queryIdx, m.trainIdx) for m in matches12])

### 11.2

In [ ]:
pts0 = kp0[matches01[:, 0]]
pts1 = kp1[matches01[:, 1]]

E, mask_E = cv2.findEssentialMat(pts0, pts1, K, method=cv2.RANSAC,
                                  prob=0.999, threshold=1.0)

_, R1, t1, mask_pose = cv2.recoverPose(E, pts0, pts1, K, mask=mask_E)

inlier_mask = (mask_E.ravel() == 1) & (mask_pose.ravel() == 255)
matches01 = matches01[inlier_mask]

Help on built-in function findEssentialMat:

findEssentialMat(...)
    findEssentialMat(points1, points2, cameraMatrix[, method[, prob[, threshold[, maxIters[, mask]]]]]) -> retval, mask
    .   @brief Calculates an essential matrix from the corresponding points in two images.
    .
    .   @param points1 Array of N (N \>= 5) 2D points from the first image. The point coordinates should
    .   be floating-point (single or double precision).
    .   @param points2 Array of the second image points of the same size and format as points1.
    .   @param cameraMatrix Camera intrinsic matrix \f$\cameramatrix{A}\f$ .
    .   Note that this function assumes that points1 and points2 are feature points from cameras with the
    .   same camera intrinsic matrix. If this assumption does not hold for your use case, use another
    .   function overload or #undistortPoints with `P = cv::NoArray()` for both cameras to transform image
    .   points to normalized image coordinates, which are valid for